# Build Rashomon Set for `diagonal_60x60`

This notebook loads a trained source policy for the `diagonal_60x60` layout, rolls out the source trajectory, and computes a Rashomon set from that trajectory.

In [ ]:
import os
import sys
import copy
from pathlib import Path
import pandas
import torch
import yaml

os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError("Could not locate repo root containing .git")

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from projects.safe_crl.pipelines.trajectory_retention.frozenlake.core.methods.source_train import (
    build_actor_critic,
    make_env_from_layout,
)
from projects.safe_crl.pipelines.trajectory_retention.frozenlake.core.methods.adapt_rashomon import (
    create_source_trajectory_rashomon_dataset,
    compute_rashomon_bounds,
)
from src.utils.general import sort_parameter_bounds_by_width
from projects.safe_crl.utils.rashomon_utils import plot_param_bounds, plot_param_bounds_per_checkpoint

%load_ext autoreload
%autoreload 2

print(f"Repo root: {repo_root}")

In [ ]:
layout = "diagonal_60x60"
seed = 0
device = "cpu"

pipeline_dir = repo_root / "projects/safe_crl/pipelines/trajectory_retention/frozenlake"
settings_dir = pipeline_dir / "settings"
source_env_file = settings_dir / "source_envs.yaml"
source_settings_file = settings_dir / "train_source_policy_settings.yaml"
adapt_settings_file = settings_dir / "downstream_adaptation_settings_ppo.yaml"
rashomon_settings_file = settings_dir / "downstream_adaptation_settings_rashomon.yaml"

outputs_candidates = [
    pipeline_dir / "outputs",
]

def find_source_run_dir(layout_name: str, run_seed: int) -> Path:
    for root in outputs_candidates:
        candidate = root / layout_name / f"seed_{run_seed}" / "source"
        if (candidate / "actor.pt").exists():
            return candidate
    searched = "\n".join(str(root / layout_name / f"seed_{run_seed}" / "source") for root in outputs_candidates)
    raise FileNotFoundError(f"Could not find source run dir with actor.pt. Searched:\n{searched}")

source_run_dir = find_source_run_dir(layout, seed)
artifact_dir = pipeline_dir / "notebooks/artifacts" / layout / f"seed_{seed}"
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Source checkpoint dir: {source_run_dir}")
print(f"Notebook artifact dir: {artifact_dir}")

In [ ]:
source_envs = yaml.safe_load(source_env_file.read_text(encoding="utf-8"))
source_settings = yaml.safe_load(source_settings_file.read_text(encoding="utf-8"))
adapt_settings = yaml.safe_load(adapt_settings_file.read_text(encoding="utf-8"))
rashomon_settings = yaml.safe_load(rashomon_settings_file.read_text(encoding="utf-8"))

source_cfg = source_envs[layout]
source_setting_cfg = source_settings[layout]
adapt_cfg = adapt_settings[layout]
rashomon_cfg = rashomon_settings[layout]["rashomon"]

source_map = source_cfg["env1_map"]
max_episode_steps = int(source_cfg["max_episode_steps"])
source_task_num = float(adapt_cfg.get("source_task_num", 0.0))
hidden = int(source_setting_cfg["ppo"]["hidden"])
activation = str(source_setting_cfg.get("activation", adapt_cfg.get("activation", "relu")))

# The parameter box grows with every iteration, so certified (hard-accuracy = 1.0)
# checkpoints only exist while the box is still small. For this 60x60 policy a short
# schedule with frequent checkpoints keeps several checkpoints fully certified, which is
# what the per-checkpoint visualisations below need. Increase `rashomon_n_iters` only if
# later checkpoints still certify (they will otherwise be dropped as invalid).
rashomon_n_iters = 50 # int(rashomon_cfg.get("rashomon_n_iters", 100000))
inverse_temp_start = int(rashomon_cfg.get("inverse_temp_start", 10))
inverse_temp_max = int(rashomon_cfg.get("inverse_temp_max", 1000))
rashomon_checkpoint = 10 # int(rashomon_cfg.get("rashomon_checkpoint", 100))
surrogate_aggregation = "min"

print(f"Layout: {layout}")
print(f"Source task number: {source_task_num}")
print(f"Activation: {activation} | Hidden size: {hidden}")
print(
    f"Rashomon config => n_iters={rashomon_n_iters}, inverse_temp_start={inverse_temp_start}, "
    f"inverse_temp_max={inverse_temp_max}, checkpoint={rashomon_checkpoint}, aggregation={surrogate_aggregation}"
)

In [ ]:
source_env_for_dim = make_env_from_layout(
    source_map,
    max_episode_steps,
    task_num=source_task_num,
    shaped=False,
)
obs_dim = int(source_env_for_dim.observation_space.shape[0])
n_actions = int(source_env_for_dim.action_space.n)
source_env_for_dim.close()

source_actor, _ = build_actor_critic(obs_dim=obs_dim, hidden=hidden, activation=activation)
actor_ckpt = source_run_dir / "actor.pt"
source_actor.load_state_dict(torch.load(actor_ckpt, map_location="cpu"))
source_actor = source_actor.to(device)
source_actor.eval()
source_actor_params = list(source_actor.parameters())

print(f"Loaded source actor from: {actor_ckpt}")
print(f"Observation dim: {obs_dim} | Actions: {n_actions}")

In [ ]:
source_rollout_env = make_env_from_layout(
    source_map,
    max_episode_steps,
    task_num=source_task_num,
    shaped=False,
)
rashomon_dataset, source_state_action_pairs = create_source_trajectory_rashomon_dataset(
    actor=copy.deepcopy(source_actor).cpu(),
    env=source_rollout_env,
    seed=seed,
    n_actions=n_actions,
)
source_rollout_env.close()

obs_tensor, action_mask_tensor = rashomon_dataset.tensors
print(f"Trajectory length: {len(source_state_action_pairs)}")
print(f"Rashomon dataset tensor shapes: obs={tuple(obs_tensor.shape)}, labels={tuple(action_mask_tensor.shape)}")

source_state_action_pairs[:5]

In [ ]:
from src.trainer.IntervalTrainer import IntervalTrainer

# Base model whose Rashomon set we compute (a CPU copy of the source actor).
actor = copy.deepcopy(source_actor).cpu()

if len(rashomon_dataset) == 0:
    raise RuntimeError("Rashomon dataset is empty.")

# Surrogate accuracy threshold: enough softmax mass must sit on the demonstrated action(s).
n_valid_actions = rashomon_dataset.tensors[1].sum(dim=1).tolist()
max_valid_actions = max(n_valid_actions) if n_valid_actions else 0.0
if max_valid_actions <= 0:
    raise RuntimeError("Rashomon dataset has no valid-action labels.")

surrogate_threshold = float(max_valid_actions / (1.0 + max_valid_actions))

# Pick the smallest inverse temperature for which every state already satisfies the threshold.
actor.eval()
with torch.no_grad():
    logits = actor(rashomon_dataset.tensors[0])
    action_mask = rashomon_dataset.tensors[1]

    selected_inverse_temp: int | None = None
    min_action_mass = float("-inf")
    for inverse_temp in range(inverse_temp_start, inverse_temp_max + 1):
        probs = torch.softmax(logits * inverse_temp, dim=1)
        valid_action_mass = (probs * action_mask).sum(dim=1)
        min_action_mass = float(valid_action_mass.min().item())
        if min_action_mass >= surrogate_threshold:
            selected_inverse_temp = inverse_temp
            break

    if selected_inverse_temp is None:
        raise ValueError(
            "Could not find inverse temperature satisfying surrogate threshold. "
            f"Best min valid-action mass={min_action_mass:.6f} < threshold={surrogate_threshold:.6f}",
        )

# Compute the Rashomon set: the parameter box that keeps 100% (hard) accuracy on the trajectory.
cert_min_threshold = 1.0
interval_trainer = IntervalTrainer(
    model=actor,
    accuracy=cert_min_threshold,
    seed=seed,
    n_iters=rashomon_n_iters,
    min_acc_increment=0.05,
    checkpoint=rashomon_checkpoint,
)
interval_trainer.compute_rashomon_set(
    dataset=rashomon_dataset,
    temperatures={None: 1.0 / selected_inverse_temp},
)

# One certificate list per checkpoint; keep the checkpoints that certify full hard accuracy.
cert_values = [
    min((c.min_hard_acc for c in certs), default=float("-inf"))
    for certs in interval_trainer.certificates
]
valid_indices = [i for i, cert in enumerate(cert_values) if cert >= cert_min_threshold]
if not valid_indices:
    best_cert = max(cert_values) if cert_values else float("-inf")
    raise ValueError(
        f"No Rashomon certificate satisfied cert_min_threshold={cert_min_threshold:.3f}. "
        f"Best certificate={best_cert:.6f}",
    )

print(f"Selected inverse temperature: {selected_inverse_temp}")
print(f"Surrogate threshold: {surrogate_threshold:.6f}")
print(f"Valid checkpoints (hard acc = 1.0): {valid_indices}")

In [ ]:
valid_bounds = [interval_trainer.bounds[i] for i in valid_indices]
print(f"Number of valid certificates with threshold >= {cert_min_threshold}: {len(valid_bounds)}")

final_certificate_idx = valid_indices[-1]
final_bounded_model = interval_trainer.bounds[final_certificate_idx]
final_param_bounds_l = [p.detach().cpu() for p in final_bounded_model.param_l]
final_param_bounds_u = [p.detach().cpu() for p in final_bounded_model.param_u]

In [ ]:
sorted_final_param_bounds = sort_parameter_bounds_by_width(final_param_bounds_l, final_param_bounds_u)
pandas.DataFrame(sorted_final_param_bounds).round(2)

## Visualise the Rashomon set

### Rashomon set of one certificate

In [ ]:
param_indices = [
    (sorted_final_param_bounds[0]['tensor_index'], sorted_final_param_bounds[0]['flat_index']),
    (sorted_final_param_bounds[1]['tensor_index'], sorted_final_param_bounds[1]['flat_index']),
]
source_params = (
    source_actor_params[param_indices[0][0]].flatten()[param_indices[0][1]].item(),
    source_actor_params[param_indices[1][0]].flatten()[param_indices[1][1]].item(),
)

plot_param_bounds(
    param_lower_bounds=final_param_bounds_l,
    param_upper_bounds=final_param_bounds_u,
    param_indices=param_indices,
    scatter_points=[
        {
            'coordinates': source_params,
            'color': 'tab:orange',
            'label': 'Source policy'
        },
    ],
    # source_params=source_params,
)

In [ ]:
param_indices = [
    (sorted_final_param_bounds[0]['tensor_index'], sorted_final_param_bounds[0]['flat_index']),
    (sorted_final_param_bounds[1]['tensor_index'], sorted_final_param_bounds[1]['flat_index']),
    (sorted_final_param_bounds[2]['tensor_index'], sorted_final_param_bounds[2]['flat_index']),
]
source_params = (
    source_actor_params[param_indices[0][0]].flatten()[param_indices[0][1]].item(),
    source_actor_params[param_indices[1][0]].flatten()[param_indices[1][1]].item(),
    source_actor_params[param_indices[2][0]].flatten()[param_indices[2][1]].item(),
)

plot_param_bounds(
    param_lower_bounds=final_param_bounds_l,
    param_upper_bounds=final_param_bounds_u,
    param_indices=param_indices,
    scatter_points=[
        {
            'coordinates': source_params,
            'color': 'tab:orange',
            'label': 'Source policy'
        },
    ]
)

### Rashomon set per checkpoint

In [ ]:
param_bounds_l_per_checkpoint = [
    [p.detach().cpu() for p in interval_trainer.bounds[i].param_l]
    for i in valid_indices
]
param_bounds_u_per_checkpoint = [
    [p.detach().cpu() for p in interval_trainer.bounds[i].param_u]
    for i in valid_indices
]

In [ ]:
### 2D:
param_indices = [
    (sorted_final_param_bounds[0]["tensor_index"], sorted_final_param_bounds[0]["flat_index"]),
    (sorted_final_param_bounds[1]["tensor_index"], sorted_final_param_bounds[1]["flat_index"]),
]
source_params = (
    source_actor_params[param_indices[0][0]].flatten()[param_indices[0][1]].item(),
    source_actor_params[param_indices[1][0]].flatten()[param_indices[1][1]].item(),
)

plot_param_bounds_per_checkpoint(
    param_bounds_l_per_checkpoint=param_bounds_l_per_checkpoint,
    param_bounds_u_per_checkpoint=param_bounds_u_per_checkpoint,
    param_indices=param_indices,
    scatter_points=[
        {
            'coordinates': source_params,
            'color': 'tab:orange',
            'label': 'Source policy'
        }
    ],
    num_checkpoints_to_plot=5,
    n_rows=1,
)

In [ ]:
### 3D:
param_indices_3d = [
    (sorted_final_param_bounds[0]["tensor_index"], sorted_final_param_bounds[0]["flat_index"]),
    (sorted_final_param_bounds[1]["tensor_index"], sorted_final_param_bounds[1]["flat_index"]),
    (sorted_final_param_bounds[2]["tensor_index"], sorted_final_param_bounds[2]["flat_index"]),
]
source_params_3d = (
    source_actor_params[param_indices_3d[0][0]].flatten()[param_indices_3d[0][1]].item(),
    source_actor_params[param_indices_3d[1][0]].flatten()[param_indices_3d[1][1]].item(),
    source_actor_params[param_indices_3d[2][0]].flatten()[param_indices_3d[2][1]].item(),
)
plot_param_bounds_per_checkpoint(
    param_bounds_l_per_checkpoint=param_bounds_l_per_checkpoint,
    param_bounds_u_per_checkpoint=param_bounds_u_per_checkpoint,
    param_indices=param_indices_3d,
    scatter_points=[
        {
            'coordinates': source_params_3d,
            'color': 'tab:orange',
            'label': 'Source policy'
        }
    ],
    num_checkpoints_to_plot=4,
    n_rows=1,
)

### New Rashomon set

In [ ]:
# Count scalar parameters represented by the bounded model
if "final_bounded_model" in globals():
    model_to_count = final_bounded_model
elif "bounded_model" in globals():
    model_to_count = bounded_model
elif "valid_bounds" in globals() and len(valid_bounds) > 0:
    model_to_count = valid_bounds[-1]
else:
    raise RuntimeError("No bounded model found. Run the Rashomon-bound computation first.")

if hasattr(model_to_count, "param_l"):
    n_params = sum(p.numel() for p in model_to_count.param_l)
elif hasattr(model_to_count, "parameters"):
    n_params = sum(p.numel() for p in model_to_count.parameters())
else:
    raise TypeError("Unsupported bounded model type for parameter counting.")

print(f"Number of parameters in bounded model: {n_params}")

In [ ]:
import copy
import torch

new_base_actor = copy.deepcopy(source_actor).to(device)
with torch.no_grad():
    for p, l, u in zip(new_base_actor.parameters(), final_param_bounds_l, final_param_bounds_u):
        l = l.to(p.device).view_as(p)
        u = u.to(p.device).view_as(p)
        choose_u = torch.rand_like(l) < 0.5
        p.copy_(torch.where(choose_u, u, l))
new_base_actor.eval()

In [ ]:
cert_min_threshold = 1.0
interval_trainer2 = IntervalTrainer(
    model=new_base_actor,
    accuracy=cert_min_threshold,
    seed=seed,
    n_iters=rashomon_n_iters,
    min_acc_increment=0.05,
    checkpoint=rashomon_checkpoint,
)
interval_trainer2.compute_rashomon_set(
    dataset=rashomon_dataset,
    temperatures={None: 1.0 / selected_inverse_temp},
)

cert_values2 = [
    min((c.min_hard_acc for c in certs), default=float("-inf"))
    for certs in interval_trainer2.certificates
]
valid_indices2 = [i for i, cert in enumerate(cert_values2) if cert >= cert_min_threshold]
if not valid_indices2:
    best_cert2 = max(cert_values2) if cert_values2 else float("-inf")
    raise ValueError(
        f"No Rashomon certificate satisfied cert_min_threshold={cert_min_threshold:.3f}. "
        f"Best certificate={best_cert2:.6f}",
    )

In [ ]:
final_certificate_idx2 = valid_indices2[-1]
final_bounded_model2 = interval_trainer2.bounds[final_certificate_idx2]
final_param_bounds_l2 = [p.detach().cpu() for p in final_bounded_model2.param_l]
final_param_bounds_u2 = [p.detach().cpu() for p in final_bounded_model2.param_u]

sorted_final_param_bounds2 = sort_parameter_bounds_by_width(final_param_bounds_l2, final_param_bounds_u2)
pandas.DataFrame(sorted_final_param_bounds2).round(2)

In [ ]:
param_indices2 = [
    (sorted_final_param_bounds2[0]['tensor_index'], sorted_final_param_bounds2[0]['flat_index']),
    (sorted_final_param_bounds2[1]['tensor_index'], sorted_final_param_bounds2[1]['flat_index']),
]
source_params = (
    source_actor_params[param_indices2[0][0]].flatten()[param_indices2[0][1]].item(),
    source_actor_params[param_indices2[1][0]].flatten()[param_indices2[1][1]].item(),
)
new_base_params = (
    list(new_base_actor.parameters())[param_indices2[0][0]].flatten()[param_indices2[0][1]].item(),
    list(new_base_actor.parameters())[param_indices2[1][0]].flatten()[param_indices2[1][1]].item(),
)
scatter_points = [
    {
        'coordinates': source_params,
        'color': 'tab:orange',
        'label': 'Source Parameters'
    },
    {
        'coordinates': new_base_params,
        'color': 'tab:green',
        'label': 'New Base Parameters'
    }
]

plot_param_bounds(
    param_lower_bounds=final_param_bounds_l2,
    param_upper_bounds=final_param_bounds_u2,
    param_indices=param_indices2,
    scatter_points=scatter_points,
)

In [ ]:
from projects.safe_crl.utils.rashomon_utils import create_rashomon_set_video
# create_rashomon_set_video(
#     param_bounds_l_per_checkpoint=param_bounds_l_per_checkpoint,
#     param_bounds_u_per_checkpoint=param_bounds_u_per_checkpoint,
#     param_indices=param_indices2,
#     scatter_points=scatter_points,
#     output_path=artifact_dir / "rashomon_set_evolution.mp4",
# )

### Two Rashomon sets

In [ ]:
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

# Use the two most uncertain shared parameters from the first Rashomon set
param_indices = [
    (sorted_final_param_bounds[0]["tensor_index"], sorted_final_param_bounds[0]["flat_index"]),
    (sorted_final_param_bounds[1]["tensor_index"], sorted_final_param_bounds[1]["flat_index"]),
]

def _get_param_value(params, tensor_idx, flat_idx):
    return params[tensor_idx].flatten()[flat_idx].item()

def _get_bounds(bounded_model, tensor_idx, flat_idx):
    lower = bounded_model.param_l[tensor_idx].detach().cpu().flatten()[flat_idx].item()
    upper = bounded_model.param_u[tensor_idx].detach().cpu().flatten()[flat_idx].item()
    return lower, upper

p1 = param_indices[0]
p2 = param_indices[1]

source_point = (
    _get_param_value(source_actor_params, *p1),
    _get_param_value(source_actor_params, *p2),
)
new_base_point = (
    _get_param_value(list(new_base_actor.parameters()), *p1),
    _get_param_value(list(new_base_actor.parameters()), *p2),
)

m1_x = _get_bounds(final_bounded_model, *p1)
m1_y = _get_bounds(final_bounded_model, *p2)
m2_x = _get_bounds(final_bounded_model2, *p1)
m2_y = _get_bounds(final_bounded_model2, *p2)

fig, ax = plt.subplots(figsize=(6, 6))

ax.add_patch(Rectangle(
    (m1_x[0], m1_y[0]),
    m1_x[1] - m1_x[0],
    m1_y[1] - m1_y[0],
    fill=True,
    edgecolor="tab:blue",
    facecolor="tab:blue",
    alpha=0.25,
    linewidth=2,
    label="Rashomon set 1",
))

ax.add_patch(Rectangle(
    (m2_x[0], m2_y[0]),
    m2_x[1] - m2_x[0],
    m2_y[1] - m2_y[0],
    fill=True,
    edgecolor="tab:green",
    facecolor="tab:green",
    alpha=0.25,
    linewidth=2,
    label="Rashomon set 2",
))

ax.scatter(*source_point, color="tab:orange", s=60, zorder=5, label="Source policy")
ax.scatter(*new_base_point, color="tab:red", s=60, zorder=5, label="New base policy")

ax.set_xlabel(f"Parameter {p1}")
ax.set_ylabel(f"Parameter {p2}")
ax.set_title("Two Rashomon sets in one subplot")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

### Union of Rashomon sets

In [ ]:
import torch

assert len(final_param_bounds_l) == len(final_param_bounds_l2)
new_final_param_bounds_l = [
    torch.minimum(a, b.to(a.device))
    for a, b in zip(final_param_bounds_l, final_param_bounds_l2)
]

new_final_param_bounds_u = [
    torch.maximum(a, b.to(a.device))
    for a, b in zip(final_param_bounds_u, final_param_bounds_u2)
]

In [ ]:
# Two widest scalar parameters of the union (updated) Rashomon set.
sorted_union_bounds = sort_parameter_bounds_by_width(new_final_param_bounds_l, new_final_param_bounds_u)
first_param_info = sorted_union_bounds[0]
second_param_info = sorted_union_bounds[1]
first_param_info, second_param_info

In [ ]:
new_first_param_lower = new_final_param_bounds_l[first_param_info['tensor_index']].flatten()[first_param_info['flat_index']].item()
new_first_param_upper = new_final_param_bounds_u[first_param_info['tensor_index']].flatten()[first_param_info['flat_index']].item()
new_second_param_lower = new_final_param_bounds_l[second_param_info['tensor_index']].flatten()[second_param_info['flat_index']].item()
new_second_param_upper = new_final_param_bounds_u[second_param_info['tensor_index']].flatten()[second_param_info['flat_index']].item()

In [ ]:
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

# Plot for the two widest parameters of the union set.
p1 = (first_param_info["tensor_index"], first_param_info["flat_index"])
p2 = (second_param_info["tensor_index"], second_param_info["flat_index"])

first_param_source_value = source_actor_params[p1[0]].flatten()[p1[1]].item()
second_param_source_value = source_actor_params[p2[0]].flatten()[p2[1]].item()

# Bounds of the previous (first) Rashomon set for the same two parameters.
prev_first = (
    final_param_bounds_l[p1[0]].flatten()[p1[1]].item(),
    final_param_bounds_u[p1[0]].flatten()[p1[1]].item(),
)
prev_second = (
    final_param_bounds_l[p2[0]].flatten()[p2[1]].item(),
    final_param_bounds_u[p2[0]].flatten()[p2[1]].item(),
)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(
    first_param_source_value,
    second_param_source_value,
    color="tab:orange",
    marker="o",
    s=50,
    label="Source policy",
    zorder=5,
)

# Previous Rashomon set (set 1).
ax.add_patch(Rectangle(
    (prev_first[0], prev_second[0]),
    prev_first[1] - prev_first[0],
    prev_second[1] - prev_second[0],
    fill=True,
    edgecolor="tab:blue",
    facecolor="tab:blue",
    linewidth=1.5,
    alpha=0.2,
    label="Previous Rashomon set",
))

# Updated (union) Rashomon set.
ax.add_patch(Rectangle(
    (new_first_param_lower, new_second_param_lower),
    new_first_param_upper - new_first_param_lower,
    new_second_param_upper - new_second_param_lower,
    fill=True,
    edgecolor="tab:green",
    facecolor="tab:green",
    linewidth=1.5,
    alpha=0.4,
    label="Updated Rashomon set",
))

ax.set_xlabel(f"Parameter {p1}")
ax.set_ylabel(f"Parameter {p2}")
ax.set_title("Union (updated) vs previous Rashomon set")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## OLD

In [ ]:
(
    param_bounds_l,
    param_bounds_u,
    bounded_model,
    selected_inverse_temp,
    surrogate_threshold,
    cert_values,
    selected_cert_idx,
) = compute_rashomon_bounds(
    actor=copy.deepcopy(source_actor).cpu(),
    rashomon_dataset=rashomon_dataset,
    seed=seed,
    rashomon_n_iters=rashomon_n_iters,
    aggregation=surrogate_aggregation,
    inverse_temp_start=inverse_temp_start,
    inverse_temp_max=inverse_temp_max,
    checkpoint=rashomon_checkpoint,
)

# print(f"Selected inverse temperature: {selected_inverse_temp}")
# print(f"Surrogate threshold: {surrogate_threshold:.6f}")
# print(f"Selected certificate index/value: {selected_cert_idx} / {cert_values[selected_cert_idx]:.6f}")
# print(f"Number of parameter tensors bounded: {len(param_bounds_l)}")

In [ ]:
rashomon_dataset_path = artifact_dir / "rashomon_dataset.pt"
state_action_pairs_path = artifact_dir / "source_policy_state_action_pairs.yaml"
bounded_model_path = artifact_dir / "bounded_model.pt"
param_bounds_path = artifact_dir / "param_bounds.pt"
summary_path = artifact_dir / "run_summary.yaml"

torch.save(rashomon_dataset, rashomon_dataset_path)
torch.save(bounded_model, bounded_model_path)
torch.save(
    {
        "param_bounds_l": [p.detach().cpu() for p in param_bounds_l],
        "param_bounds_u": [p.detach().cpu() for p in param_bounds_u],
    },
    param_bounds_path,
)
state_action_pairs_path.write_text(
    yaml.safe_dump(source_state_action_pairs, sort_keys=False),
    encoding="utf-8",
)

summary = {
    "layout": layout,
    "seed": int(seed),
    "source_checkpoint_dir": str(source_run_dir),
    "source_actor_checkpoint": str(actor_ckpt),
    "dataset_size": int(len(rashomon_dataset)),
    "trajectory_steps": int(len(source_state_action_pairs)),
    "rashomon_n_iters": int(rashomon_n_iters),
    "surrogate_aggregation": surrogate_aggregation,
    "inverse_temp_start": int(inverse_temp_start),
    "inverse_temp_max": int(inverse_temp_max),
    "selected_inverse_temp": int(selected_inverse_temp),
    "surrogate_threshold": float(surrogate_threshold),
    "selected_certificate_index": int(selected_cert_idx),
    "selected_certificate": float(cert_values[selected_cert_idx]),
    "all_certificates": [float(v) for v in cert_values],
    "rashomon_dataset_path": str(rashomon_dataset_path),
    "source_policy_state_action_pairs_path": str(state_action_pairs_path),
    "bounded_model_path": str(bounded_model_path),
    "param_bounds_path": str(param_bounds_path),
}
summary_path.write_text(yaml.safe_dump(summary, sort_keys=False), encoding="utf-8")

print(f"Saved rashomon dataset: {rashomon_dataset_path}")
print(f"Saved source state-action pairs: {state_action_pairs_path}")
print(f"Saved bounded model: {bounded_model_path}")
print(f"Saved param bounds: {param_bounds_path}")
print(f"Saved summary: {summary_path}")